In [2]:
import pandas as pd
import numpy as np
import re #text cleaning
from difflib import SequenceMatcher #similarity between text values

df = pd.read_csv("Alumni_Data_1000_Rows.csv")
df

,Alumni_ID,First_Name,Last_Name,Graduation_Year,Degree_Earned,Major_Academic_Program,Current_Company,Job_Title,Email_Address,Phone_Number,Brochure_Feature_Consent
0,ALU0001,Varun,Patel,2015,MBA,Artificial Intelligence and Machine Learning,Deloitte,Full Stack Developer,varun.patel1@outlook.com,+91 7440213415,Yes
1,ALU0002,Swathi,Shetty,2015,B.Tech,Computer Science (Data Science),Capgemini,Full Stack Developer,swathi.shetty2@gmail.com,+91 9410529190,Yes
2,ALU0003,Anusha,Das,2021,B.E.,Civil Engineering,Cognizant Technology Solutions,DevOps Engineer,anusha.das3@gmail.com,+91 7685731524,No
3,ALU0004,Nikhil,Iyer,2017,B.E.,Mechanical Engineering,Accenture,Data Analyst,nikhil.iyer4@proton.me,+91 7415393687,No
4,ALU0005,Pooja,Chowdhury,2019,B.Tech,Civil Engineering,PwC,Data Scientist,pooja.chowdhury5@proton.me,+91 7338444264,No
...,...,...,...,...,...,...,...,...,...,...,...
995,ALU0996,Sanjay,Mishra,2015,B.E.,Computer Science and Engineering,Bosch,Data Scientist,sanjay.mishra996@yahoo.com,+91 8015314465,No
996,ALU0997,Madhav,Verma,2024,B.Tech,Computer Science and Engineering,Cognizant,Systems Engineer,madhav.verma997@proton.me,+91 8273950560,Yes
997,ALU0998,Bhavana,Mehta,2018,MBA,Computer Science and Engineering,Deloitte,Technical Consultant,bhavana.mehta998@proton.me,+91 8411712184,Yes
998,ALU0999,Manoj,Gupta,2025,B.Tech,Computer Engineering,Amazon,Cloud Engineer,manoj.gupta999@gmail.com,+91 7983590548,No


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 1000
Number of columns: 11


In [4]:
df.columns.tolist()

['Alumni_ID',
 'First_Name',
 'Last_Name',
 'Graduation_Year',
 'Degree_Earned',
 'Major_Academic_Program',
 'Current_Company',
 'Job_Title',
 'Email_Address',
 'Phone_Number',
 'Brochure_Feature_Consent']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Alumni_ID                 1000 non-null   object
 1   First_Name                1000 non-null   object
 2   Last_Name                 1000 non-null   object
 3   Graduation_Year           1000 non-null   int64 
 4   Degree_Earned             1000 non-null   object
 5   Major_Academic_Program    1000 non-null   object
 6   Current_Company           1000 non-null   object
 7   Job_Title                 1000 non-null   object
 8   Email_Address             1000 non-null   object
 9   Phone_Number              1000 non-null   object
 10  Brochure_Feature_Consent  1000 non-null   object
dtypes: int64(1), object(10)
memory usage: 86.1+ KB


In [6]:
missing_values = df.isnull().sum()

print(missing_values)

Alumni_ID                   0
First_Name                  0
Last_Name                   0
Graduation_Year             0
Degree_Earned               0
Major_Academic_Program      0
Current_Company             0
Job_Title                   0
Email_Address               0
Phone_Number                0
Brochure_Feature_Consent    0
dtype: int64


In [7]:
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

Alumni_ID                   0.0
First_Name                  0.0
Last_Name                   0.0
Graduation_Year             0.0
Degree_Earned               0.0
Major_Academic_Program      0.0
Current_Company             0.0
Job_Title                   0.0
Email_Address               0.0
Phone_Number                0.0
Brochure_Feature_Consent    0.0
dtype: float64


In [8]:
print("Duplicate Alumni IDs:", df["Alumni_ID"].duplicated().sum())

Duplicate Alumni IDs: 0


In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [11]:
text_columns = [
    "first_name",
    "last_name",
    "degree_earned",
    "major_academic_program",
    "current_company",
    "job_title",
    "email_address",
    "phone_number",
    "brochure_feature_consent"
]
## "  B.Tech  " -> B.Tech // removes extra spaces
for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [12]:
df["first_name"] = df["first_name"].str.title()
df["last_name"] = df["last_name"].str.title()

In [13]:
print(df["degree_earned"].value_counts())

degree_earned
B.E.      173
B.Sc      172
B.Tech    170
MBA       166
MCA       162
M.Tech    157
Name: count, dtype: int64


In [14]:
degree_mapping = {
    "B.E": "B.E.",
    "BE": "B.E.",
    "B.E.": "B.E.",
    "BTECH": "B.Tech",
    "B TECH": "B.Tech",
    "B.Tech": "B.Tech",
    "BSC": "B.Sc",
    "B.Sc": "B.Sc",
    "MTECH": "M.Tech",
    "M TECH": "M.Tech",
    "M.Tech": "M.Tech",
    "MCA": "MCA",
    "MBA": "MBA"
}

df["degree_earned"] = (
    df["degree_earned"]
    .str.strip()
    .map(degree_mapping)
    .fillna(df["degree_earned"])
)

In [15]:
print(df["major_academic_program"].value_counts())

major_academic_program
Information Technology                          110
Mechanical Engineering                          108
Civil Engineering                               106
Computer Science (Data Science)                 105
Artificial Intelligence and Machine Learning    102
Business Administration                         102
Computer Science and Engineering                102
Computer Engineering                             93
Electrical and Electronics Engineering           90
Electronics and Communication Engineering        82
Name: count, dtype: int64


In [16]:
df["major_academic_program"] = (
    df["major_academic_program"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [17]:
df["current_company"] = (
    df["current_company"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [19]:
df["job_title"] = (
    df["job_title"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(df["job_title"].value_counts())

job_title
Technical Consultant         63
Machine Learning Engineer    62
Software Engineer            58
Data Analyst                 57
Associate Consultant         56
Business Analyst             56
UI Developer                 54
Senior Software Engineer     53
Product Analyst              52
Systems Engineer             52
Database Administrator       50
Full Stack Developer         48
Project Engineer             47
Data Scientist               47
Backend Developer            44
QA Engineer                  43
DevOps Engineer              42
Cloud Engineer               41
Software Developer           40
Frontend Developer           35
Name: count, dtype: int64


In [20]:
df["email_address"] = (
    df["email_address"]
    .str.strip()
    .str.lower()
)

In [21]:
df["phone_number"] = (
    df["phone_number"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [22]:
print(df["brochure_feature_consent"].value_counts())

brochure_feature_consent
No     521
Yes    479
Name: count, dtype: int64


In [24]:
df["brochure_feature_consent"] = (
    df["brochure_feature_consent"]
    .str.strip()
    .str.title()
)
consent_mapping = {
    "Yes": 1,
    "No": 0
}

df["brochure_feature_consent_encoded"] = (
    df["brochure_feature_consent"]
    .map(consent_mapping)
)

In [25]:
print(df["graduation_year"].dtype)

int64


In [26]:
df["graduation_year"] = pd.to_numeric(
    df["graduation_year"],
    errors="coerce"
)

In [27]:
print(df["graduation_year"].isnull().sum())

0


In [28]:
print("Minimum graduation year:", df["graduation_year"].min())
print("Maximum graduation year:", df["graduation_year"].max())

Minimum graduation year: 2015
Maximum graduation year: 2025


In [29]:
invalid_years = df[
    (df["graduation_year"] < 1900) |
    (df["graduation_year"] > 2026)
]

print(invalid_years)

Empty DataFrame
Columns: [alumni_id, first_name, last_name, graduation_year, degree_earned, major_academic_program, current_company, job_title, email_address, phone_number, brochure_feature_consent, brochure_feature_consent_encoded]
Index: []


In [30]:
df["full_name"] = (
    df["first_name"] + " " + df["last_name"]
)

In [31]:
df[["first_name", "last_name", "full_name"]].head()

,first_name,last_name,full_name
0,Varun,Patel,Varun Patel
1,Swathi,Shetty,Swathi Shetty
2,Anusha,Das,Anusha Das
3,Nikhil,Iyer,Nikhil Iyer
4,Pooja,Chowdhury,Pooja Chowdhury


In [32]:
df["job_title_normalized"] = (
    df["job_title"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [33]:
df["major_normalized"] = (
    df["major_academic_program"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^a-z0-9\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

In [37]:
df.head()



Shape: (1000, 15)
Missing values:
alumni_id                           0
first_name                          0
last_name                           0
graduation_year                     0
degree_earned                       0
major_academic_program              0
current_company                     0
job_title                           0
email_address                       0
phone_number                        0
brochure_feature_consent            0
brochure_feature_consent_encoded    0
full_name                           0
job_title_normalized                0
major_normalized                    0
dtype: int64


In [35]:
df.info

<bound method DataFrame.info of     alumni_id first_name  last_name  graduation_year degree_earned  \
0     ALU0001      Varun      Patel             2015           MBA   
1     ALU0002     Swathi     Shetty             2015        B.Tech   
2     ALU0003     Anusha        Das             2021          B.E.   
3     ALU0004     Nikhil       Iyer             2017          B.E.   
4     ALU0005      Pooja  Chowdhury             2019        B.Tech   
..        ...        ...        ...              ...           ...   
995   ALU0996     Sanjay     Mishra             2015          B.E.   
996   ALU0997     Madhav      Verma             2024        B.Tech   
997   ALU0998    Bhavana      Mehta             2018           MBA   
998   ALU0999      Manoj      Gupta             2025        B.Tech   
999   ALU1000     Nikhil      Verma             2018           MCA   

                           major_academic_program  \
0    Artificial Intelligence and Machine Learning   
1                 Com

In [38]:
print("Shape:", df.shape)
print("Missing values:")
print(df.isnull().sum())

Shape: (1000, 15)
Missing values:
alumni_id                           0
first_name                          0
last_name                           0
graduation_year                     0
degree_earned                       0
major_academic_program              0
current_company                     0
job_title                           0
email_address                       0
phone_number                        0
brochure_feature_consent            0
brochure_feature_consent_encoded    0
full_name                           0
job_title_normalized                0
major_normalized                    0
dtype: int64


In [ ]:
## Similarity Function

In [39]:
def text_similarity(text1, text2):
    """
    Calculates similarity between two text strings.
    Returns a value between 0 and 1.
    """
    
    text1 = str(text1).lower().strip()
    text2 = str(text2).lower().strip()
    
    return SequenceMatcher(None, text1, text2).ratio()

In [40]:
print(text_similarity(
    "software engineer",
    "software engineer"
))

1.0


In [41]:
print(text_similarity(
    "software engineer",
    "software developer"
))

0.7428571428571429


In [42]:
def categorical_similarity(value1, value2):
    return 1.0 if value1 == value2 else 0.0

In [44]:
print(categorical_similarity("B.Tech", "B.Tech"))

1.0


In [45]:
print(categorical_similarity("B.Tech", "MBA"))

0.0


In [ ]:
## Graduation-Year Similarity

In [46]:
def graduation_similarity(year1, year2):
    difference = abs(year1 - year2)
    
    if difference == 0:
        return 1.0
    elif difference <= 2:
        return 0.75
    elif difference <= 5:
        return 0.50
    else:
        return 0.25

In [47]:
print(graduation_similarity(2023, 2023))
print(graduation_similarity(2023, 2021))
print(graduation_similarity(2023, 2018))

1.0
0.75
0.5


In [ ]:
## Adapted Weighted Score

In [52]:
def calculate_alumni_score(student, alumnus):
    
    role_score = text_similarity(
        student["target_role"],
        alumnus["job_title"]
    )
    
    major_score = text_similarity(
        student["major"],
        alumnus["major_academic_program"]
    )
    
    degree_score = categorical_similarity(
        student["degree"],
        alumnus["degree_earned"]
    )
    
    graduation_score = graduation_similarity(
        student["graduation_year"],
        alumnus["graduation_year"]
    )
    
    final_score = (
        0.40 * role_score +
        0.30 * major_score +
        0.15 * degree_score +
        0.15 * graduation_score
    )
    
    return final_score

In [54]:
student = {
    "target_role": "Machine Learning Engineer",
    "major": "Computer Science and Engineering",
    "degree": "B.Tech",
    "graduation_year": 2026
}

In [55]:
df["match_score"] = df.apply(
    lambda row: calculate_alumni_score(student, row),
    axis=1
)

In [56]:
df["match_percentage"] = (
    df["match_score"] * 100
).round(2)

In [57]:
top_5_alumni = (
    df.sort_values(
        by="match_score",
        ascending=False
    )
    .head(5)
)

In [58]:
df = df.sort_values(
    by="match_score",
    ascending=False
)

In [59]:
df[
    [
        "alumni_id",
        "full_name",
        "job_title",
        "major_academic_program",
        "match_percentage"
    ]
].head(10)

,alumni_id,full_name,job_title,major_academic_program,match_percentage
932,ALU0933,Swathi Patel,Machine Learning Engineer,Computer Science and Engineering,88.75
882,ALU0883,Nandini Singh,Machine Learning Engineer,Computer Science and Engineering,88.75
533,ALU0534,Rahul Naidu,Machine Learning Engineer,Computer Engineering,85.58
374,ALU0375,Anusha Mehta,Machine Learning Engineer,Mechanical Engineering,82.92
748,ALU0749,Rohit Gupta,Machine Learning Engineer,Computer Engineering,81.83
151,ALU0152,Lakshmi Kumar,Machine Learning Engineer,Civil Engineering,79.64
746,ALU0747,Deepak Singh,Machine Learning Engineer,Computer Science and Engineering,77.50
152,ALU0153,Anusha Shetty,Machine Learning Engineer,Electrical and Electronics Engineering,75.89
247,ALU0248,Aarav Singh,Systems Engineer,Computer Science and Engineering,75.76
996,ALU0997,Madhav Verma,Systems Engineer,Computer Science and Engineering,75.76


In [60]:
processed_df = df.copy()

In [61]:
processed_columns = [
    "alumni_id",
    "full_name",
    "first_name",
    "last_name",
    "graduation_year",
    "degree_earned",
    "major_academic_program",
    "current_company",
    "job_title",
    "email_address",
    "phone_number",
    "brochure_feature_consent",
    "brochure_feature_consent_encoded",
    "match_score",
    "match_percentage"
]

processed_df = processed_df[processed_columns]

In [62]:
processed_df.to_csv(
    "alumni_processed.csv",
    index=False
)

In [63]:
processed_df.to_json(
    "alumni.json",
    orient="records",
    indent=4
)

In [64]:
final_df = pd.read_json("alumni.json")

print("Final shape:", final_df.shape)

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nDuplicate rows:")
print(final_df.duplicated().sum())

print("\nDuplicate Alumni IDs:")
print(final_df["alumni_id"].duplicated().sum())

Final shape: (1000, 15)

Missing values:
alumni_id                           0
full_name                           0
first_name                          0
last_name                           0
graduation_year                     0
degree_earned                       0
major_academic_program              0
current_company                     0
job_title                           0
email_address                       0
phone_number                        0
brochure_feature_consent            0
brochure_feature_consent_encoded    0
match_score                         0
match_percentage                    0
dtype: int64

Duplicate rows:
0

Duplicate Alumni IDs:
0
